In [1]:
## imports
import pandas as pd
import numpy as np
import re
import requests
import yaml


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 1. Example 1: no credentials; no wrapper

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

## 1.1 Query to pull some data

In [2]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

In [4]:
## use requests to call the api
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))

## get the json contents of the response 
## here, we're assuming valid response
naep_resp_j = naep_resp.json()
naep_resp_j

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

<Response [200]>

<class 'requests.models.Response'>


{'status': 200,
 'serviceVersion': '10.24.2025.2',
 'dwellTimeMS': '343.7818',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_MEMORY',
 'Source': 'B11B',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDi

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


## 1.2 What happens if there's an error in our query?

In [5]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [6]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

<Response [400]>

In [6]:
## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
#naep_wrong_resp.json() # uncomment to see error

In [7]:
naep_wrong_resp.text

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2704\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2325\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 923\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 349"}'

### 1.2.2 More all-purpose way of allowing remainder of calls to run: try, except

In [8]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except Exception as e:
    print('Failed to get result from API due to error:')
    print(e) # or just: pass

Failed to get result from API due to error:
Invalid control character at: line 1 column 293 (char 292)


### 1.2.3 Can usually also find more targeted way but that varies more across APIs

In [9]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along those lines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2704\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2325\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 923\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 349"}'

'{"status":200,"serviceVersion":"10.24.2025.2","dwellTimeMS":"343.7818","avgWebHostCPUTotalLoad":"N/A","dataHitType":"FROM_MEMORY","Source":"B11B","result": [{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"1","varValueLabel":"Male","value":139.099504632971,"isStatDisplayable":1,"errorFlag":0},{"year":2011,"sample":"R3","yearSampleLabel":"2011","Cohort":2,"CohortLabel":"Grade 8","stattype":"MN:MN","subject":"WRI","grade":8,"scale":"WRIRP","jurisdiction":"NP","jurisLabel":"National public","variable":"GENDER","variableLabel":"Sex","varValue":"2","varValueLabel":"Female","value":158.567104984955,"isStatDisplayable":1,"errorFlag":0}]}'

NAEP results not found


## Activity 1: writing a function to make multiple, sequential calls

- Say we want to pull the data for grades 4, 8, and 12
- How can we write a function that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [10]:
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')

In [11]:
def naep_api_calls(grades: list,
                   baseurl = 'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?',
                   query_part1 = 'type=data&subject=writing',
                   query_part3 = '&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'):
                       
    '''This function calls the NAEP API to get data on writing scores for the grade requested in 2011.
    
    Parameters:
        baseurl (str): default URL for NAEP or similar API
        query_part1 (str): first batch of arguments
        grades (list of int): what grades to get data on
        query_part3 (str): third batch of arguments
    Returns:
        DataFrame with the results'''
    
    dflist = []
    
    for grade in grades:
        naep_query = baseurl + query_part1  + query_part3 + ('&grade=' + str(grade))
        naep_resp = requests.get(naep_query)
        
        #if "System.Exception" in naep_resp.text:
        #    print(f"NAEP results not found for grade {str(grade)}, please try with grade in range of 1-12")
        #else:
        try:
            naep_resp_df = pd.DataFrame(naep_resp.json()['result'])
            dflist.append(naep_resp_df)
        except Exception as e:
            print("Failed to get result from API for grade {} due to error:".format(str(grade)))
            print(e)
        
    combined_result_df = pd.concat(dflist)
    
    return(combined_result_df)


naep_api_calls(grades = [4,8,12])

Failed to get result from API for grade 4 due to error:
Invalid control character at: line 1 column 293 (char 292)


,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0
0,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,1,Male,141.256978,1,0
1,2011,R3,2011,3,Grade 12,MN:MN,WRI,12,WRIRP,NP,National public,GENDER,Sex,2,Female,155.385917,1,0


# 2. Example 2: needs credentials; no wrapper

In [ ]:
# Format example

In [12]:
API_KEY = "Your Key"
API_KEY = "rAZn1RX6Tr2o999Ly8EvD6ofsNGBZg3CMDsDxAwKGTEdWIi1zm9g23h_1ll5oix2Z0hSDiRloteTewU35nBqydGvoq0LngcGlZPP6LERVywiS_uc18KMmtvg9qQuZXYx"

In [13]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Hanover,NH,03755"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [14]:
yelp_genjson["businesses"]

[{'id': 'JFE0XffhpP3Bi3DQRvymVg',
  'alias': 'little-havana-hanover',
  'name': 'Little Havana',
  'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/941vaRo98hcxAOpN7KICSA/o.jpg',
  'is_closed': False,
  'url': 'https://www.yelp.com/biz/little-havana-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=ZmitoFfF-pb6UbUwNTC3_A',
  'review_count': 23,
  'categories': [{'alias': 'cuban', 'title': 'Cuban'},
   {'alias': 'latin', 'title': 'Latin American'}],
  'rating': 4.9,
  'coordinates': {'latitude': 43.700743, 'longitude': -72.287599},
  'transactions': ['delivery', 'pickup'],
  'location': {'address1': '15 Lebanon St',
   'address2': None,
   'address3': None,
   'city': 'Hanover',
   'zip_code': '03755',
   'country': 'US',
   'state': 'NH',
   'display_address': ['15 Lebanon St', 'Hanover, NH 03755']},
  'phone': '+18383831000',
  'display_phone': '(838) 383-1000',
  'distance': 102.83322895097157},
 {'id': '8ybF6YyR

In [15]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf.head()

{'id': 'JFE0XffhpP3Bi3DQRvymVg',
 'alias': 'little-havana-hanover',
 'name': 'Little Havana',
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/941vaRo98hcxAOpN7KICSA/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/little-havana-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=ZmitoFfF-pb6UbUwNTC3_A',
 'review_count': 23,
 'categories': [{'alias': 'cuban', 'title': 'Cuban'},
  {'alias': 'latin', 'title': 'Latin American'}],
 'rating': 4.9,
 'coordinates': {'latitude': 43.700743, 'longitude': -72.287599},
 'transactions': ['delivery', 'pickup'],
 'location': {'address1': '15 Lebanon St',
  'address2': None,
  'address3': None,
  'city': 'Hanover',
  'zip_code': '03755',
  'country': 'US',
  'state': 'NH',
  'display_address': ['15 Lebanon St', 'Hanover, NH 03755']},
 'phone': '+18383831000',
 'display_phone': '(838) 383-1000',
 'distance': 102.83322895097157}

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,location,phone,display_phone,distance,price
0,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,False,https://www.yelp.com/biz/little-havana-hanover...,23,"[{'alias': 'cuban', 'title': 'Cuban'}, {'alias...",4.9,"{'latitude': 43.700743, 'longitude': -72.287599}","[delivery, pickup]","{'address1': '15 Lebanon St', 'address2': None...",+18383831000,(838) 383-1000,102.833229,NaN
1,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,False,https://www.yelp.com/biz/mollys-restaurant-and...,577,"[{'alias': 'tradamerican', 'title': 'American'...",3.9,"{'latitude': 43.701144, 'longitude': -72.2894249}",[delivery],"{'address1': '43 South Main St', 'address2': '...",+16036432570,(603) 643-2570,250.830160,$$
2,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/FbA7IU...,False,https://www.yelp.com/biz/base-camp-cafe-hanove...,262,"[{'alias': 'himalayan', 'title': 'Himalayan/Ne...",4.4,"{'latitude': 43.700626, 'longitude': -72.2887803}",[delivery],"{'address1': '3 Lebanon St', 'address2': 'Ste ...",+16036432007,(603) 643-2007,196.139758,$$
3,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,False,https://www.yelp.com/biz/bistro-at-six-hanover...,2,"[{'alias': 'lounges', 'title': 'Lounges'}, {'a...",4.0,"{'latitude': 43.7001146, 'longitude': -72.2877...",[],"{'address1': '6 E South St', 'address2': None,...",+16036430600,(603) 643-0600,198.651788,$$
4,KA8yhrd-ClVYMyOefXdVYg,lous-restaurant-and-bakery-hanover,Lou's Restaurant & Bakery,https://s3-media0.fl.yelpcdn.com/bphoto/ZguQWu...,False,https://www.yelp.com/biz/lous-restaurant-and-b...,452,"[{'alias': 'tradamerican', 'title': 'American'...",4.2,"{'latitude': 43.70143, 'longitude': -72.289001}",[delivery],"{'address1': '30 S Main St', 'address2': '', '...",+16036433321,(603) 643-3321,245.079232,$$


In [16]:
## more data-specific way of summarizing
## we're doing a simple approach and just retaining
## cols that have a simple str structure
## if doing for real, would want to extract things
def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)


,id,alias,name,image_url,url,phone,display_phone,price
JFE0XffhpP3Bi3DQRvymVg,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,https://www.yelp.com/biz/little-havana-hanover...,+18383831000,(838) 383-1000,NaN
8ybF6YyRldtZmU9jil4xlg,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,https://www.yelp.com/biz/mollys-restaurant-and...,+16036432570,(603) 643-2570,$$
XVGEEIH5rVB2QzW-qywcJw,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/FbA7IU...,https://www.yelp.com/biz/base-camp-cafe-hanove...,+16036432007,(603) 643-2007,$$
wyV_NfYn4ZOfp_sHMDPcAw,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,https://www.yelp.com/biz/bistro-at-six-hanover...,+16036430600,(603) 643-0600,$$
KA8yhrd-ClVYMyOefXdVYg,KA8yhrd-ClVYMyOefXdVYg,lous-restaurant-and-bakery-hanover,Lou's Restaurant & Bakery,https://s3-media0.fl.yelpcdn.com/bphoto/ZguQWu...,https://www.yelp.com/biz/lous-restaurant-and-b...,+16036433321,(603) 643-3321,$$
5WW4g_LRwau29KyjZGLyAA,5WW4g_LRwau29KyjZGLyAA,sawtooth-kitchen-hanover,Sawtooth Kitchen,https://s3-media0.fl.yelpcdn.com/bphoto/61MNG4...,https://www.yelp.com/biz/sawtooth-kitchen-hano...,+16036435134,(603) 643-5134,NaN
1Q9gTry0GH2NFA7O398xeA,1Q9gTry0GH2NFA7O398xeA,casa-brava-tapas-hanover,Casa Brava Tapas,https://s3-media0.fl.yelpcdn.com/bphoto/fA3__V...,https://www.yelp.com/biz/casa-brava-tapas-hano...,+16038509763,(603) 850-9763,NaN


# Activity 2: pull restaurants in a different location

- Try running a business search query for your hometown or another place by constructing a query similar to `yelp_genquery` but changing the location parameter
- Other endpoints require feeding what's called the business' fusion id into the API. Take an id from `yelp_stronly.id` and use the documentation here to pull the reviews for that business: https://www.yelp.com/developers/documentation/v3/business_reviews
- **Challenge**: generalize the previous step by writing a function that (1) takes a list of business ids as an input, (2) calls the reviews API for each id, (3) returns the results, and (4) rowbinds all results, i.e. turns them into a single, usable DataFrame

In [51]:
# change location
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "bubble tea"

my_location_tpe = "Taipei,Taiwan,03456"

yelp_genquery_tpe = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location_tpe)

## use requests to call the API
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp_tpe = requests.get(yelp_genquery_tpe, headers = header)
# yelp_genresp_tpe

## then, look at structure of response
yelp_genjson_tpe = yelp_genresp_tpe.json()

## turn JSON into usable data (DF)
yelp_gendf_tpe = pd.DataFrame(yelp_genjson_tpe['businesses'])
#list(yelp_gendf_tpe) # list columns
yelp_gendf_tpe['location1'] = yelp_gendf_tpe.location.apply(lambda loclist: loclist['address1'])
yelp_gendf_tpe[['alias', 'name', 'url', 'review_count', 'rating', 'location1', 'price']]

# Optional: Restrict to cols that are just strings
#yelp_stronly_tpe = [clean_yelp_json(one_b) for one_b in yelp_genjson_tpe['businesses']]
#yelp_stronly_tpe_df = pd.concat(yelp_stronly_tpe)

#yelp_stronly_tpe_df.head(10)

,alias,name,url,review_count,rating,location1,price
0,50嵐-萬華區,50嵐,https://www.yelp.com/biz/50%E5%B5%90-%E8%90%AC...,21,4.2,西寧南路48之4號,$
1,幸福堂-萬華區,Xing Fu Tang,https://www.yelp.com/biz/%E5%B9%B8%E7%A6%8F%E5...,58,4.1,成都路29號,$$
2,老虎堂-中正區,Tiger Sugar,https://www.yelp.com/biz/%E8%80%81%E8%99%8E%E5...,19,4.7,南陽街1號,$
3,coco都可-台北市萬華區-2,Coco,https://www.yelp.com/biz/coco%E9%83%BD%E5%8F%A...,8,4.6,武昌街二段4號,$
4,50嵐-台北市萬華區-6,50 Lan,https://www.yelp.com/biz/50%E5%B5%90-%E5%8F%B0...,3,4.0,漢中街136號,$$
5,可不可熟成紅茶-大安區,Kebuke,https://www.yelp.com/biz/%E5%8F%AF%E4%B8%8D%E5...,4,4.5,師大路119號,NaN
6,春水堂-台北市中正區-2,Chun Shui Tang,https://www.yelp.com/biz/%E6%98%A5%E6%B0%B4%E5...,34,4.2,忠孝西路一段66號B2樓,$$
7,coco都可茶飲成都店-萬華區,Coco,https://www.yelp.com/biz/coco%E9%83%BD%E5%8F%A...,4,4.3,成都路41號,$
8,珍煮丹-大安區,Truedan,https://www.yelp.com/biz/%E7%8F%8D%E7%85%AE%E4...,13,4.9,信義路三段200號,$
9,樺達奶茶-中正區-2,樺達奶茶,https://www.yelp.com/biz/%E6%A8%BA%E9%81%94%E5...,10,4.5,忠孝西路一段47號,$


In [29]:
# change location
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"

my_location_tpe = "Kuala Lumpur,Malaysia"
yelp_genquery_tpe = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location_tpe)

## use requests to call the API
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp_tpe = requests.get(yelp_genquery_tpe, headers = header)
# yelp_genresp_tpe

## then, look at structure of response
yelp_genjson_tpe = yelp_genresp_tpe.json()

## turn JSON into usable data (DF)
yelp_gendf_tpe = pd.DataFrame(yelp_genjson_tpe['businesses'])
#list(yelp_gendf_tpe) # list columns
yelp_gendf_tpe['location1'] = yelp_gendf_tpe.location.apply(lambda loclist: loclist['address1'])
yelp_gendf_tpe[['alias', 'name', 'url', 'review_count', 'rating', 'location1', 'price']]

# Optional: Restrict to cols that are just strings
#yelp_stronly_tpe = [clean_yelp_json(one_b) for one_b in yelp_genjson_tpe['businesses']]
#yelp_stronly_tpe_df = pd.concat(yelp_stronly_tpe)

#yelp_stronly_tpe_df.head(10)

,alias,name,url,review_count,rating,location1,price
0,restaurant-sun-huat-kee-kuala-lumpur,Restaurant Sun Huat Kee,https://www.yelp.com/biz/restaurant-sun-huat-k...,3,5.0,"348, Jalan Tun Abdul Razak",NaN
1,nasi-lemak-antarabangsa-kuala-lumpur,Nasi Lemak Antarabangsa,https://www.yelp.com/biz/nasi-lemak-antarabang...,3,4.7,4 Jalan Raja Muda Musa,$
2,ho-kow-hainanese-kopitiam-kuala-lumpur,Ho Kow Hainanese Kopitiam,https://www.yelp.com/biz/ho-kow-hainanese-kopi...,9,4.6,"1, Jalan Balai Polis",NaN
3,the-orchid-conservatory-kuala-lumpur,The Orchid Conservatory,https://www.yelp.com/biz/the-orchid-conservato...,3,4.7,The Majestic Hotel,NaN
4,baba-nyonya-kuala-lumpur,Baba Nyonya,https://www.yelp.com/biz/baba-nyonya-kuala-lum...,2,5.0,156 Jalan Ampang,NaN
5,restoran-han-kee-kuala-lumpur,Restoran Han Kee,https://www.yelp.com/biz/restoran-han-kee-kual...,2,5.0,"N0. 42, Jalan Sultan",NaN
6,ah-loong-grilled-seafood-kuala-lumpur,Ah Loong Grilled Seafood,https://www.yelp.com/biz/ah-loong-grilled-seaf...,1,5.0,"12-14, Jalan Jinjang Aman 2",NaN
7,kedai-makanan-yut-kee-kuala-lumpur,Kedai Makanan Yut Kee,https://www.yelp.com/biz/kedai-makanan-yut-kee...,29,4.1,"1, Jalan Kamunting",$
8,the-canteen-room-kuala-lumpur,The Canteen Room,https://www.yelp.com/biz/the-canteen-room-kual...,2,5.0,A-0-2 Wisma HB Megan Phileo Avenue,NaN
9,bijan-kuala-lumpur,Bijan,https://www.yelp.com/biz/bijan-kuala-lumpur?ad...,26,4.4,3 Jalan Ceylon,$$$


In [33]:
# change location
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"

my_location_tpe = "London, UK"
yelp_genquery_tpe = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location_tpe)

## use requests to call the API
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp_tpe = requests.get(yelp_genquery_tpe, headers = header)
# yelp_genresp_tpe

## then, look at structure of response
yelp_genjson_tpe = yelp_genresp_tpe.json()

## turn JSON into usable data (DF)
yelp_gendf_tpe = pd.DataFrame(yelp_genjson_tpe['businesses'])
#list(yelp_gendf_tpe) # list columns
yelp_gendf_tpe['location1'] = yelp_gendf_tpe.location.apply(lambda loclist: loclist['address1'])
yelp_gendf_tpe[['alias', 'name', 'url', 'review_count', 'rating', 'location1', 'price']]

# Optional: Restrict to cols that are just strings
#yelp_stronly_tpe = [clean_yelp_json(one_b) for one_b in yelp_genjson_tpe['businesses']]
#yelp_stronly_tpe_df = pd.concat(yelp_stronly_tpe)

#yelp_stronly_tpe_df.head(10)

,alias,name,url,review_count,rating,location1,price
0,the-mayfair-chippy-london-2,The Mayfair Chippy,https://www.yelp.com/biz/the-mayfair-chippy-lo...,693,4.5,14 North Audley Street,££
1,dishoom-london,Dishoom,https://www.yelp.com/biz/dishoom-london?adjust...,3013,4.5,12 Upper Saint Martin's Lane,£££
2,flat-iron-london-2,Flat Iron,https://www.yelp.com/biz/flat-iron-london-2?ad...,545,4.4,17 Beak Street,££
3,the-victoria-paddington,The Victoria,https://www.yelp.com/biz/the-victoria-paddingt...,346,4.4,10a Strathearn Place,££
4,blacklock-london-6,Blacklock,https://www.yelp.com/biz/blacklock-london-6?ad...,194,4.6,The Basement,£££
5,dishoom-london-7,Dishoom,https://www.yelp.com/biz/dishoom-london-7?adju...,866,4.5,22 Kingly Street,££
6,the-alchemist-london-4,The Alchemist,https://www.yelp.com/biz/the-alchemist-london-...,100,4.3,63-66 St Martin's Ln,££
7,mother-mash-london,Mother Mash,https://www.yelp.com/biz/mother-mash-london?ad...,645,4.3,26 Ganton Street,££
8,the-barbary-london,The Barbary,https://www.yelp.com/biz/the-barbary-london?ad...,124,4.8,16 Neal's Yard,£££
9,restaurant-gordon-ramsay-london-3,Restaurant Gordon Ramsay,https://www.yelp.com/biz/restaurant-gordon-ram...,281,4.3,68 Royal Hospital Road,££££


In [35]:
# change location
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"

my_location_tpe = "Hanover,NH,03755"
yelp_genquery_tpe = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location_tpe)

## use requests to call the API
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp_tpe = requests.get(yelp_genquery_tpe, headers = header)
# yelp_genresp_tpe

## then, look at structure of response
yelp_genjson_tpe = yelp_genresp_tpe.json()

## turn JSON into usable data (DF)
yelp_gendf_tpe = pd.DataFrame(yelp_genjson_tpe['businesses'])
#list(yelp_gendf_tpe) # list columns
yelp_gendf_tpe['location1'] = yelp_gendf_tpe.location.apply(lambda loclist: loclist['address1'])
yelp_gendf_tpe[['alias', 'name', 'url', 'review_count', 'rating', 'location1', 'price']]

# Optional: Restrict to cols that are just strings
#yelp_stronly_tpe = [clean_yelp_json(one_b) for one_b in yelp_genjson_tpe['businesses']]
#yelp_stronly_tpe_df = pd.concat(yelp_stronly_tpe)

#yelp_stronly_tpe_df.head(10)

,alias,name,url,review_count,rating,location1,price
0,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://www.yelp.com/biz/mollys-restaurant-and...,577,3.9,43 South Main St,$$
1,base-camp-cafe-hanover,Base Camp Cafe,https://www.yelp.com/biz/base-camp-cafe-hanove...,262,4.4,3 Lebanon St,$$
2,little-havana-hanover,Little Havana,https://www.yelp.com/biz/little-havana-hanover...,23,4.9,15 Lebanon St,NaN
3,bistro-at-six-hanover,Bistro at Six,https://www.yelp.com/biz/bistro-at-six-hanover...,2,4.0,6 E South St,$$
4,sawtooth-kitchen-hanover,Sawtooth Kitchen,https://www.yelp.com/biz/sawtooth-kitchen-hano...,39,4.2,33 S Main St,NaN
5,lous-restaurant-and-bakery-hanover,Lou's Restaurant & Bakery,https://www.yelp.com/biz/lous-restaurant-and-b...,452,4.2,30 S Main St,$$
6,casa-brava-tapas-hanover,Casa Brava Tapas,https://www.yelp.com/biz/casa-brava-tapas-hano...,7,4.7,6 South St,NaN
7,pine-restaurant-hanover-2,PINE Restaurant,https://www.yelp.com/biz/pine-restaurant-hanov...,250,3.5,2 E Wheelock St,$$$
8,tuk-tuk-thai-cuisine-hanover,Tuk Tuk Thai Cuisine,https://www.yelp.com/biz/tuk-tuk-thai-cuisine-...,340,4.2,5 S Main St,$$
9,murphy-s-on-the-green-hanover,Murphy’s on the Green,https://www.yelp.com/biz/murphy-s-on-the-green...,205,3.7,11 S Main St,$$


In [43]:
# Get one random business
one_biz = yelp_stronly_df.sample()[['name', 'id']]
one_biz = yelp_gendf_tpe.iloc[8]

In [42]:
url = "https://api.yelp.com/v3/businesses/Ecdu5qYM09F647Uoez99vg/reviews?"
# header = {‘Authorization’: f”Bearer {API_KEY}""}
response = requests.get(url, headers=header)
print(response.text)


{"reviews": [{"id": "0t6MRc_rBZrrK8W6gcSIcQ", "url": "https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=0t6MRc_rBZrrK8W6gcSIcQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A", "text": "Ramunto's has the best pizza in the Upper Valley. (The NY style meatball/extra cheese pizza is our family fave.) We've been eating at Ramunto's for almost...", "rating": 5.0, "time_created": "2026-01-13 09:25:22", "user": {"id": "0HDmIFddGAD0Xs3g85XwxA", "profile_url": "https://www.yelp.com/user_details?userid=0HDmIFddGAD0Xs3g85XwxA", "image_url": null, "name": "Novel N."}}, {"id": "KMBiy0t5YTnmvLMR2i60rA", "url": "https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=KMBiy0t5YTnmvLMR2i60rA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A", "text": "I order online the meditterran pizza added shrimp. 

In [39]:
response.text

'{"reviews": [{"id": "8I8ALslMWvOQmd7AnO4N8w", "url": "https://www.yelp.com/biz/the-shed-restaurant-plainview-plainview?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=8I8ALslMWvOQmd7AnO4N8w&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=AYAiHNSGxz_RRHzq3cO46w", "text": "I LOVE THE SHED!\\n\\nAny location doesn\'t matter my experience has blown me away and left me satisfied.\\n\\nThe grilled chicken entree was better than my grilled...", "rating": 5, "time_created": "2023-09-23 14:26:01", "user": {"id": "HOxaVDup_Ctag_npGDdF0A", "profile_url": "https://www.yelp.com/user_details?userid=HOxaVDup_Ctag_npGDdF0A", "image_url": "https://s3-media3.fl.yelpcdn.com/photo/P78IjRv3JfpOcjE0T1dgqw/o.jpg", "name": "Emily B."}}, {"id": "sPyEsPtJz48wyOd3DIrtGA", "url": "https://www.yelp.com/biz/the-shed-restaurant-plainview-plainview?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=sPyEsPtJz48wyOd3DIrtGA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=AYAiHNSGxz_RR

In [47]:
one_biz.id

'vMyN7JL5cJExJORgIobbQg'

In [48]:
# look at reviews of business with this id
base_url_reviews = f'https://api.yelp.com/v3/businesses/{one_biz.id}/reviews'
yelp_genquery_reviews = (base_url_reviews)

## use requests to call the API
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp_reviews = requests.get(yelp_genquery_reviews, headers = header)
yelp_genresp_reviews

## then, look at structure of response
yelp_genjson_reviews = yelp_genresp_reviews.json()
yelp_genjson_reviews

<Response [200]>

{'reviews': [{'id': 'f5-39FJC7UbTb2rtBIbhyg',
   'url': 'https://www.yelp.com/biz/tuk-tuk-thai-cuisine-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=f5-39FJC7UbTb2rtBIbhyg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A',
   'text': 'Special curry avocado curry with chicken     Appetizer    Very very good portion  sized good',
   'rating': 5.0,
   'time_created': '2026-02-03 18:15:32',
   'user': {'id': 'sezXZQlcHeTb2CxMlxZKXA',
    'profile_url': 'https://www.yelp.com/user_details?userid=sezXZQlcHeTb2CxMlxZKXA',
    'image_url': 'https://s3-media0.fl.yelpcdn.com/photo/gOpS778gW7ePiE4KaZ1d7A/o.jpg',
    'name': 'George G.'}},
  {'id': 'wCdsj52xvhr-T8jU5y2xdg',
   'url': 'https://www.yelp.com/biz/tuk-tuk-thai-cuisine-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=wCdsj52xvhr-T8jU5y2xdg&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A',
   'text': '3.5 Stars. Decent Thai flavors. We got 

In [41]:
yelp_genjson_reviews['reviews']

[{'id': 'UpqByKb8iBIFzYvFmvtt0A',
  'url': 'https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=UpqByKb8iBIFzYvFmvtt0A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=AYAiHNSGxz_RRHzq3cO46w',
  'text': "Best pizza ever from this townie. Even other towns nearby... still can't get me to fully change my mind on how this place has the best sauce, pies, and noms...",
  'rating': 5,
  'time_created': '2023-08-08 06:53:00',
  'user': {'id': 'kgAiZU60RefGdjkeRdi0pA',
   'profile_url': 'https://www.yelp.com/user_details?userid=kgAiZU60RefGdjkeRdi0pA',
   'image_url': 'https://s3-media2.fl.yelpcdn.com/photo/lLzLXgaHEtgaPwhPVfs74g/o.jpg',
   'name': 'Mia A.'}},
 {'id': 'Yxj8gWX2isGf0-DVBPq67A',
  'url': 'https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=Yxj8gWX2isGf0-DVBPq67A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=AYAiHNSGxz_R

In [45]:
yelp_genjson_reviews

{'reviews': [{'id': 'UpqByKb8iBIFzYvFmvtt0A',
   'url': 'https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=UpqByKb8iBIFzYvFmvtt0A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=AYAiHNSGxz_RRHzq3cO46w',
   'text': "Best pizza ever from this townie. Even other towns nearby... still can't get me to fully change my mind on how this place has the best sauce, pies, and noms...",
   'rating': 5,
   'time_created': '2023-08-08 06:53:00',
   'user': {'id': 'kgAiZU60RefGdjkeRdi0pA',
    'profile_url': 'https://www.yelp.com/user_details?userid=kgAiZU60RefGdjkeRdi0pA',
    'image_url': 'https://s3-media2.fl.yelpcdn.com/photo/lLzLXgaHEtgaPwhPVfs74g/o.jpg',
    'name': 'Mia A.'}},
  {'id': 'Yxj8gWX2isGf0-DVBPq67A',
   'url': 'https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=AYAiHNSGxz_RRHzq3cO46w&hrid=Yxj8gWX2isGf0-DVBPq67A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&u

In [18]:
## turn JSON into usable data (DF)
yelp_gendf_reviews = pd.DataFrame(yelp_genjson_reviews['reviews'])
yelp_gendf_reviews

,id,url,text,rating,time_created,user
0,ZKw3pgptzE3aKiKmwFLK1Q,https://www.yelp.com/biz/sushiya-hanover-2?adj...,Just had an phenomenal meal here and look forw...,5,2022-08-25 19:19:55,"{'id': '76gvqovFuTpEoOHysjbY6w', 'profile_url'..."
1,nzGzOK01t_DztjuYGwB4SQ,https://www.yelp.com/biz/sushiya-hanover-2?adj...,This restaurant is now takeout only. The sushi...,3,2022-10-17 18:15:08,"{'id': 'trIbciU6PXfiXr0BYz0DQw', 'profile_url'..."
2,xfBpLVOxGF9UZAoe8wQpBw,https://www.yelp.com/biz/sushiya-hanover-2?adj...,Ordered takeout for 6 rolls. All of them taste...,4,2021-10-14 19:20:23,"{'id': 'Imrkj1Ywx5RVNDmDh9AFcw', 'profile_url'..."
